<a href="https://colab.research.google.com/github/Sapikzzz/AI-ASL/blob/main/Transer_with_zero_weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
grassknoted_asl_alphabet_path = kagglehub.dataset_download('grassknoted/asl-alphabet')

print('Data source import complete.')


In [ ]:
import numpy as np
import os
import cv2
from sklearn.utils import shuffle
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import gc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization
import kagglehub
import seaborn as sns
import pandas as pd
import skimage
from skimage.transform import resize
from sklearn.metrics import classification_report, confusion_matrix
import os
from glob import glob
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

2025-05-28 12:19:06.564706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748434746.795961      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748434746.873506      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
# Download latest version
grassknoted_asl_alphabet_path = kagglehub.dataset_download("grassknoted/asl-alphabet")
print(grassknoted_asl_alphabet_path)
train_path = os.path.join(grassknoted_asl_alphabet_path, '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train')

/kaggle/input/asl-alphabet


In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices()

tf.test.is_gpu_available()

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            print("GPU configured successfully")
    except RuntimeError as e:
        print("Error configuring GPU:", e)

Num GPUs Available:  1
GPU configured successfully


I0000 00:00:1748434761.120461      35 gpu_device.cc:2022] Created device /device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [ ]:
target_size = (96, 96)
target_dims = (96, 96, 3)
n_classes = 29
val_frac = 0.1
batch_size = 256

data_augmentor = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,
    rescale=1./255,
    validation_split=val_frac
)

In [ ]:
train_gen = data_augmentor.flow_from_directory(train_path, target_size=target_size, batch_size=batch_size, shuffle=True, subset='training')
val_gen = data_augmentor.flow_from_directory(train_path, target_size=target_size, batch_size=batch_size, subset='validation')

Found 78300 images belonging to 29 classes.
Found 8700 images belonging to 29 classes.


In [ ]:
MobileNet_layers = MobileNetV2(weights='imagenet', include_top=False, input_shape=target_dims)

I0000 00:00:1748434802.504972      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = Sequential([
    MobileNet_layers,
    GlobalAveragePooling2D(),
    Dense(1024, activation='relu'),
    Dropout(0.5),
    Dense(n_classes, activation='softmax')
])

In [ ]:
for layer in MobileNet_layers.layers:
    if hasattr(layer, 'kernel_initializer'):
        layer.kernel.assign(layer.kernel_initializer(layer.kernel.shape))

In [ ]:
for layer in MobileNet_layers.layers:
    layer.trainable = True

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
MobileNet_history = model.fit(train_gen, epochs=5, validation_data=val_gen)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


I0000 00:00:1748434850.208530     108 service.cc:148] XLA service 0x7e809c001ee0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748434850.209270     108 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1748434854.166284     108 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1748434860.212443     108 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748434860.403551     108 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748434860.663052     108 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748434860.8694

185/306 ━━━━━━━━━━━━━━━━━━━━ 3:51 2s/step - accuracy: 0.3565 - loss: 2.2370

E0000 00:00:1748435243.824535     107 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748435244.031244     107 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748435244.248288     107 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748435244.438680     107 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748435249.482947     107 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:0

306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4790 - loss: 1.7770

E0000 00:00:1748435576.672678     106 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748435576.878735     106 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748435577.095370     106 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1748435577.286289     106 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


306/306 ━━━━━━━━━━━━━━━━━━━━ 770s 2s/step - accuracy: 0.4797 - loss: 1.7741 - val_accuracy: 0.0345 - val_loss: 10.2912
Epoch 2/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 342s 1s/step - accuracy: 0.9548 - loss: 0.1371 - val_accuracy: 0.0345 - val_loss: 10.1154
Epoch 3/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.9781 - loss: 0.0701 - val_accuracy: 0.0345 - val_loss: 9.8726
Epoch 4/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.9832 - loss: 0.0544 - val_accuracy: 0.0345 - val_loss: 15.3589
Epoch 5/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 325s 1s/step - accuracy: 0.9861 - loss: 0.0457 - val_accuracy: 0.0345 - val_loss: 19.2306


In [ ]:
model.save('/MobileNet_model.keras')